# Pragma Model — LoRA Fine-Tuning

**Purpose:** Fine-tune Phi-3-mini-4k-instruct on Pragma ethical reasoning training data using LoRA/QLoRA.

**Runtime:** Google Colab free T4 GPU (~90 min for 3 epochs on ~140 examples)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `ml/data/train.jsonl` and `ml/data/eval.jsonl` to this Colab session
3. Add your HuggingFace write token to Colab Secrets (key: `HF_TOKEN`)
4. Set your HF repo name in the config cell below (e.g. `yourname/pragma-ethics-v1`)

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.11.0 \
    datasets accelerate bitsandbytes huggingface_hub
print('✅ Dependencies installed')

In [ ]:
# ── Step 2: Config — EDIT THIS ────────────────────────────────────────────────
HF_REPO     = 'YOUR_HF_USERNAME/pragma-ethics-v1'   # <-- change this
BASE_MODEL  = 'microsoft/Phi-3-mini-4k-instruct'
TRAIN_FILE  = 'train.jsonl'
EVAL_FILE   = 'eval.jsonl'
NUM_EPOCHS  = 3
BATCH_SIZE  = 2
GRAD_ACCUM  = 8      # effective batch = 16
MAX_SEQ_LEN = 1024
LR          = 2e-4
OUTPUT_DIR  = './pragma-checkpoints'

print(f'Config OK — will push to: {HF_REPO}')

In [ ]:
# ── Step 3: Authenticate with HuggingFace ────────────────────────────────────
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('✅ HuggingFace login successful')

In [ ]:
# ── Step 4: Verify GPU ────────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Step 5: Load and preview training data ────────────────────────────────────
import json
from datasets import Dataset

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_records = load_jsonl(TRAIN_FILE)
eval_records  = load_jsonl(EVAL_FILE)
print(f'Train: {len(train_records)} examples')
print(f'Eval:  {len(eval_records)} examples')
print(f'\nSample categories: {set(r["meta"]["category"] for r in train_records[:20])}')

In [ ]:
# ── Step 6: Load tokenizer and model (4-bit quantized) ───────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading tokenizer from {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model (4-bit quantized)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('✅ Model loaded')

In [ ]:
# ── Step 7: Apply LoRA adapter ────────────────────────────────────────────────
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect ~1-2% trainable — rest frozen

In [ ]:
# ── Step 8: Build datasets ────────────────────────────────────────────────────
def record_to_text(record):
    msgs = record['messages']
    if hasattr(tokenizer, 'apply_chat_template'):
        try:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        except Exception:
            pass
    # Fallback for Phi-3 format
    parts = []
    for m in msgs:
        parts.append(f"<|{m['role']}|>\n{m['content']}<|end|>")
    return '\n'.join(parts)

train_ds = Dataset.from_dict({'text': [record_to_text(r) for r in train_records]})
eval_ds  = Dataset.from_dict({'text': [record_to_text(r) for r in eval_records]})

# Quick sanity check — print first 200 chars of one example
print(train_ds[0]['text'][:200])
print(f'\n✅ Datasets ready — train={len(train_ds)}, eval={len(eval_ds)}')

In [ ]:
# ── Step 9: Train ─────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    learning_rate=LR,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=5,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,
    push_to_hub=True,
    hub_model_id=HF_REPO,
    hub_token=HF_TOKEN,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    tokenizer=tokenizer,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete')

In [ ]:
# ── Step 10: Push to HuggingFace Hub ─────────────────────────────────────────
print(f'Pushing model to {HF_REPO}...')
trainer.push_to_hub()
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f'\n✅ Model live at: https://huggingface.co/{HF_REPO}')
print(f'\nNext step — set this in Railway dashboard:')
print(f'  CUSTOM_MODEL_REPO = {HF_REPO}')

In [ ]:
# ── Step 11 (optional): Quick inference test before deploying ─────────────────
from peft import PeftModel
from transformers import pipeline

SYSTEM_PROMPT = """You are an ethical reasoning engine. Respond with ONLY valid JSON using EXACTLY this structure:
{"kantian_analysis": "<string>", "utilitarian_analysis": "<string>", "virtue_ethics_analysis": "<string>",
 "risk_flags": ["<string>", ...], "confidence_score": <float 0-1>, "recommendation": "<string>"}
risk_flags must be a flat array chosen from: bias, fairness, discrimination, transparency, harm
Do NOT wrap in markdown. Do NOT add extra keys."""

test_messages = [
    {'role': 'system',  'content': SYSTEM_PROMPT},
    {'role': 'user',    'content': 'Category: hiring\nDecision: Reject job candidate\nContext: {"gender": "female", "experience_years": 10, "education": "master"}'}
]

pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=512)
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
output = pipe(prompt)[0]['generated_text']
print('Model output:')
print(output[len(prompt):])